In [ ]:
# hourly demand profile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1) Monthly old vs new electricity demand
monthly = pd.read_csv("/Users/zarapashov/Desktop/oxford data passiv /plotting sheets - passiv house(new vs old total E requirement).csv")
# Columns: Month (Jan, Feb, ...), Old, New

# 2) Dimensionless hourly profiles (48 half-hour points per day)
profiles_48 = pd.read_csv("/Users/zarapashov/Desktop/oxford data passiv /passiv haus calculations(part aiii) Hourly demand).csv")
# Columns include: 'Settlement period', 'Aut Wd','Aut Sat','Aut Sun',
#                  'Smr Wd','Smr Sat','Smr Sun','Spr Wd','Spr Sat','Spr Sun',
#                  'Wtr Wd','Wtr Sat','Wtr Sun', etc.

# 3) Yearly calendar for 2025
cal = pd.read_csv("/Users/zarapashov/Desktop/oxford data passiv /passiv haus calculations(yearly calendar).csv")
# Columns: Date, Month, Season, Daytype, Day of week

In [ ]:
# Parse Date and drop empty rows
cal["Date"] = pd.to_datetime(cal["Date"], errors="coerce")
cal = cal[cal["Date"].notna()].copy()   # keep only real dates (365 rows)

# Merge monthly NEW demand (the Passivhaus building) onto calendar by Month text
cal = cal.merge(monthly[["Month", "New"]], on="Month", how="left")

# Days per month and daily kWh per day
cal["days_in_month"] = cal.groupby("Month")["Date"].transform("count")
cal["daily_kwh"] = cal["New"] / cal["days_in_month"]

print(cal.head(20))



In [ ]:
# Map Season/Daytype names to profile column prefixes/suffixes
season_map = {
    "Winter": "Wtr",
    "Spring": "Spr",
    "Summer": "Smr",
    "Autumn": "Aut",
}
daytype_map = {
    "Weekday": "Wd",
    "Saturday": "Sat",
    "Sunday": "Sun",
}

# Build a 24-hour normalised profile table: Season, Daytype, Hour (0–23), weight
profile_rows = []

for season_name, season_code in season_map.items():
    for daytype_name, day_code in daytype_map.items():
        col = f"{season_code} {day_code}"
        if col not in profiles_48.columns:
            continue  # just in case

        weights48 = profiles_48[col].to_numpy(dtype=float)   # 48 half-hour values

        # Collapse to 24 hours by summing each pair of half-hours
        hourly24 = np.array([weights48[2*h] + weights48[2*h+1] for h in range(24)])

        # Normalise so the 24 hours sum to 1 (dimensionless daily distribution)
        hourly24_norm = hourly24 / hourly24.sum()

        for h in range(24):
            profile_rows.append({
                "Season": season_name,
                "Daytype": daytype_name,
                "Hour": h,              # 0–23
                "weight": hourly24_norm[h],
            })

profile_hourly = pd.DataFrame(profile_rows)
print(profile_hourly)


In [ ]:
# Repeat each day 24 times to get 24 hours
cal_hourly = cal.loc[cal.index.repeat(24)].copy()
cal_hourly["Hour"] = np.tile(np.arange(24), len(cal))

# Merge in the correct hourly weight using Season + Daytype + Hour
cal_hourly = cal_hourly.merge(
    profile_hourly,
    on=["Season", "Daytype", "Hour"],
    how="left"
)

# Hourly demand in kWh
cal_hourly["demand_kwh"] = cal_hourly["daily_kwh"] * cal_hourly["weight"]

# Build datetime stamp for plotting (hour start time)
cal_hourly["datetime"] = cal_hourly["Date"] + pd.to_timedelta(cal_hourly["Hour"], unit="h")

# Sort by time just to be clean
cal_hourly = cal_hourly.sort_values("datetime").reset_index(drop=True)
 
#check
print("Annual demand from hourly profile:", cal_hourly["demand_kwh"].sum())
print("Annual demand from monthly New:", monthly["New"].sum())



In [ ]:
# 4 subplots for each season
import matplotlib.pyplot as plt
import pandas as pd

# Ensure datetime is datetime
cal_hourly["datetime"] = pd.to_datetime(cal_hourly["datetime"])

# Order of seasons
season_order = ["Winter", "Spring", "Summer", "Autumn"]

# Choose colours for each season
season_colors = {
    "Winter": "blue",
    "Spring": "green",
    "Summer": "orange",
    "Autumn": "red"
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.ravel()

for i, season in enumerate(season_order):
    ax = axes[i]
    
    # Select season data
    sdata = cal_hourly[cal_hourly["Season"] == season].copy()
    sdata = sdata.sort_values("datetime")
    
    # Take an example week from start of the season
    start = sdata["datetime"].min()
    end = start + pd.Timedelta(days=7)
    week = sdata[(sdata["datetime"] >= start) & (sdata["datetime"] < end)]
    
    # Plot with colour for that season
    ax.plot(week["datetime"], week["demand_kwh"], color=season_colors[season])
    ax.set_title(f"{season}  ")
    ax.set_xlabel("Time")
    ax.set_ylabel("Demand (kWh per hour)")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Hourly Electricity Demand – Example Week in Each Season", y=1.02)
plt.tight_layout()
plt.show()




In [ ]:
#PLOT FOR ONE SUMMER DAY/ ONE WINTER DAY (HOURLY) 
import pandas as pd
import matplotlib.pyplot as plt

# Make sure datetime is datetime
cal_hourly["datetime"] = pd.to_datetime(cal_hourly["datetime"])
cal_hourly["date_only"] = cal_hourly["datetime"].dt.date

# --- 1. Choose ONE winter day and ONE summer day ---
# Here I pick the day with the highest daily demand in each season (nice and defensible)

winter_days = (
    cal_hourly[cal_hourly["Season"] == "Winter"]
    .groupby("date_only")["demand_kwh"].sum()
)
winter_date = winter_days.idxmax()   # python date object

summer_days = (
    cal_hourly[cal_hourly["Season"] == "Summer"]
    .groupby("date_only")["demand_kwh"].sum()
)
summer_date = summer_days.idxmax()

# Slice those two days (24 rows each)
winter_day = cal_hourly[cal_hourly["date_only"] == winter_date].copy()
summer_day = cal_hourly[cal_hourly["date_only"] == summer_date].copy()

# Hour of day for x-axis
winter_day["hour"] = winter_day["datetime"].dt.hour
summer_day["hour"] = summer_day["datetime"].dt.hour

# --- 2. Plot them on subplots ---

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

axes[0].plot(winter_day["hour"], winter_day["demand_kwh"], marker="o", color="blue")
axes[0].set_title(f"Winter day ({winter_date})")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Demand (kWh per hour)")
axes[0].set_xticks(range(0, 24, 2))

axes[1].plot(summer_day["hour"], summer_day["demand_kwh"], marker="o", color="orange")
axes[1].set_title(f"Summer day ({summer_date})")
axes[1].set_xlabel("Hour of day")
axes[1].set_xticks(range(0, 24, 2))

plt.suptitle("Hourly Electricity Demand – Representative Winter vs Summer Day", y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
#ONE WINNTER WEEK/ ONE SUMMER WEEK HOURLY 
# Example winter week
winter_week = cal_hourly[
    (cal_hourly["datetime"] >= "2025-01-15") &
    (cal_hourly["datetime"] <  "2025-01-22")
]

plt.figure(figsize=(10,4))
plt.plot(winter_week["datetime"], winter_week["demand_kwh"], color='blue')
plt.ylabel("Demand (kWh per hour)")
plt.xlabel("Time")
plt.title("Hourly Demand – Example Winter Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Example summer week
summer_week = cal_hourly[
    (cal_hourly["datetime"] >= "2025-07-15") &
    (cal_hourly["datetime"] <  "2025-07-22")
]

plt.figure(figsize=(10,4))
plt.plot(summer_week["datetime"], summer_week["demand_kwh"], color='orange')
plt.ylabel("Demand (kWh per hour)")
plt.xlabel("Time")
plt.title("Hourly Demand – Example Summer Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [25]:
#SAVE DATA TO CSV

cal_hourly.to_csv("hourly_demand_profile.csv", index=False)


In [ ]:
#plot whole year

